> **Note — notebook surface is moving.** Starting with `v0.4.0`, all notebooks
> in this repository will move to the dedicated
> [`forecastability-examples`](https://github.com/example/forecastability-examples)
> sibling repository. The library itself will keep only deterministic Python
> APIs, scripts under `scripts/`, and recipe pages under `docs/recipes/`.
> See [docs/plan/v0_4_0_examples_repo_split_ultimate_plan.md](../../docs/plan/v0_4_0_examples_repo_split_ultimate_plan.md).

# Air Passengers — All-Methods Showcase

This notebook is the **product catalogue** for the `forecastability` package.  
Every diagnostic, every method, every surface — demonstrated on a single memorable series.

| Section | What it shows |
|--------:|:--------------|
| 1 | Start with a series everyone recognises |
| 2 | Fastest surface — one deterministic triage call |
| 3 | The six diagnostic lenses (F1–F6) |
| 4 | Scorer registry — same series, different lenses |
| 5 | Canonical AMI / pAMI with significance bands |
| 6 | Robustness — how stable are these results? |
| 7 | Rolling-origin evaluation |
| 8 | Batch triage — portfolio context |
| 9 | Interpretation patterns A–E |
| 10 | Exogenous driver screening |
| 11 | Agent-ready serialisation |
| 12 | Generated report |
| 13 | Product surface map — where to go next |

## Setup

In [ ]:
%matplotlib inline
import warnings

warnings.filterwarnings("ignore")

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from forecastability import (
    ForecastabilityAnalyzer,
    TriageRequest,
    default_registry,
    run_batch_triage,
    run_triage,
)
from forecastability.adapters.triage_presenter import present_triage_result
from forecastability.pipeline import (
    run_canonical_example,
    run_rolling_origin_evaluation,
)
from forecastability.reporting import build_canonical_markdown
from forecastability.services.predictive_info_learning_curve_service import build_predictive_info_learning_curve
from forecastability.services.spectral_predictability_service import build_spectral_predictability
from forecastability.triage import (
    BatchSeriesRequest,
    BatchTriageRequest,
    MethodPlan,
    build_triage_result_bundle,
)
from forecastability.use_cases.run_exogenous_screening_workbench import (
    DRIVER_SUMMARY_TABLE_COLUMNS,
    HORIZON_USEFULNESS_TABLE_COLUMNS,
    driver_summary_table_rows,
    horizon_usefulness_table_rows,
    run_exogenous_screening_workbench,
)
from forecastability.utils.config import ExogenousScreeningWorkbenchConfig
from forecastability.utils.datasets import (
    generate_ar1,
    generate_henon_map,
    generate_simulated_stock_returns,
    generate_white_noise,
    load_air_passengers,
)
from forecastability.utils.robustness import run_backend_comparison, run_sample_size_stress
from forecastability.utils.types import ExogenousBenchmarkResult

In [ ]:
# Shared palette
COLORS = {
    "ami": "#2196F3",
    "pami": "#FF9800",
    "band": "#E0E0E0",
    "sig": "#4CAF50",
    "warn": "#F44336",
    "grid": "#F5F5F5",
    "med": "#9C27B0",
}
plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

---
## 1. Start with a series everyone recognises

The **Air Passengers** dataset (Box & Jenkins, 1970) — 144 monthly totals of international
airline passengers, Jan 1949 – Dec 1960.  Trend + multiplicative seasonality makes it the
canonical time-series playground.

In [ ]:
air = load_air_passengers()
months = pd.date_range("1949-01", periods=len(air), freq="MS")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time plot
axes[0].plot(months, air, color=COLORS["ami"], linewidth=1.5)
axes[0].set_title("Air Passengers — monthly totals")
axes[0].set_ylabel("Passengers (thousands)")

# Seasonal profile
seasonal = pd.Series(air, index=months).groupby(months.month).mean()
axes[1].bar(seasonal.index, seasonal.values, color=COLORS["pami"], alpha=0.7)
axes[1].set_title("Mean seasonal profile")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Mean passengers")

fig.tight_layout()
plt.show()

print(f"Series length: {len(air)}, min: {air.min():.0f}, max: {air.max():.0f}")

---
## 2. Fastest surface — one deterministic triage call

A single `run_triage()` call answers: *is this series forecastable? How? At which horizons?*

> **Reference:** Catt (2026), *Forecastability as an Information-Theoretic Limit on Prediction.*

In [ ]:
triage_result = run_triage(
    TriageRequest(series=air, max_lag=24, random_state=42)
)

# Canonical overview table
view = present_triage_result(triage_result)
overview = {
    "readiness": view.readiness_status,
    "forecastability": view.forecastability_class,
    "directness": view.directness_class,
    "modeling_regime": view.modeling_regime,
    "primary_lags": view.primary_lags,
    "pattern_class": view.pattern_class,
    "recommendation": view.recommendation,
}
display(pd.DataFrame([overview]).T.rename(columns={0: "Air Passengers"}))

In [ ]:
# Raw / partial dependence plot
ar = triage_result.analyze_result
assert ar is not None

raw_lags = np.arange(1, len(ar.raw) + 1)
partial_lags = np.arange(1, len(ar.partial) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw_lags, ar.raw, label="AMI (raw)", color=COLORS["ami"], linewidth=2)
ax.plot(partial_lags, ar.partial, label="pAMI (partial)", color=COLORS["pami"], linewidth=2)

# Shade mediated component over the overlapping range
n_overlap = min(len(ar.raw), len(ar.partial))
overlap_lags = np.arange(1, n_overlap + 1)
ax.fill_between(
    overlap_lags, ar.raw[:n_overlap], ar.partial[:n_overlap],
    alpha=0.15, color=COLORS["med"], label="Mediated component",
)

ax.set_xlabel("Horizon h")
ax.set_ylabel("Mutual information (nats)")
ax.set_title("Air Passengers — Raw vs. Partial Dependence")
ax.legend()
plt.tight_layout()
plt.show()

> **What this tells us:** Air Passengers is a *highly forecastable* series with strong
> dependence at seasonal lags.  The gap between AMI and pAMI reveals substantial
> mediated (indirect) dependence absorbed by intermediate horizons.

---
## 3. The six diagnostic lenses (F1–F6)

Each diagnostic answers a different question about predictability.

### F1: Forecastability Profile

*At which horizons does genuine predictive information exist?*

> **Reference:** Catt (2026), *On the Limits of Prediction: Forecastability Profiles and Information Decay in Time Series.*

In [ ]:
fp = triage_result.forecastability_profile
assert fp is not None

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fp.horizons, fp.values, color=COLORS["ami"], linewidth=2, label="F(h)")
ax.axhline(
    fp.epsilon, color=COLORS["warn"], linestyle="--", alpha=0.6,
    label=f"\u03b5 = {fp.epsilon:.4f}",
)
for h in fp.informative_horizons:
    idx = fp.horizons.index(h)
    ax.plot(h, fp.values[idx], "o", color=COLORS["sig"], markersize=8, zorder=5)
ax.axvline(
    fp.peak_horizon, color=COLORS["med"], linestyle=":", alpha=0.5,
    label=f"Peak h={fp.peak_horizon}",
)
ax.set_xlabel("Horizon h")
ax.set_ylabel("Forecastability F(h)")
ax.set_title("F1: Forecastability Profile")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

display(
    Markdown(
        f"**Summary:** {fp.summary}\n\n"
        f"**Model now:** {fp.model_now}\n\n"
        f"**Review horizons:** {fp.review_horizons}\n\n"
        f"**Avoid horizons:** {fp.avoid_horizons}"
    )
)

### F2: Information Limits (Theoretical Ceiling)

*What is the maximum achievable predictive gain at each horizon?*

> **Reference:** Catt, *The Knowable Future: Mapping the Decay of Past–Future Mutual Information Across Forecast Horizons.*

In [ ]:
tld = triage_result.theoretical_limit_diagnostics
assert tld is not None

ceiling = tld.forecastability_ceiling_by_horizon
horizons_f2 = np.arange(1, len(ceiling) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(horizons_f2, 0, ceiling, alpha=0.25, color=COLORS["ami"])
ax.plot(
    horizons_f2, ceiling, color=COLORS["ami"], linewidth=2,
    label="Forecastability ceiling",
)
ax.set_xlabel("Horizon h")
ax.set_ylabel("Max predictive gain (nats)")
ax.set_title("F2: Information-Theoretic Ceiling")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

display(
    Markdown(
        f"**Ceiling summary:** {tld.ceiling_summary}\n\n"
        f"**Compression warning:** {tld.compression_warning or 'None'}\n\n"
        f"**DPI warning:** {tld.dpi_warning or 'None'}"
    )
)

### F3: Predictive Information Learning Curves

*How much history (lookback) does the series need before information saturates?*

> **Reference:** Morawski et al. (2025), *How Patterns Dictate Learnability in Sequential Data.*

In [ ]:
f3 = build_predictive_info_learning_curve(air, random_state=42)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    f3.window_sizes, f3.information_values,
    "o-", color=COLORS["ami"], linewidth=2,
)
if f3.convergence_index >= 0:
    ci = f3.convergence_index
    ax.axvline(
        f3.window_sizes[ci], color=COLORS["sig"], linestyle="--",
        label=f"Convergence at k={f3.window_sizes[ci]}",
    )
ax.set_xlabel("Lookback window k")
ax.set_ylabel("Predictive information I_pred(k)")
ax.set_title("F3: Predictive Information Learning Curve")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

display(
    Markdown(
        f"**Recommended lookback:** {f3.recommended_lookback} steps\n\n"
        f"**Plateau detected:** {f3.plateau_detected}\n\n"
        f"**Reliability warnings:** {f3.reliability_warnings or 'None'}"
    )
)

### F4: Spectral Predictability

*How much predictable (periodic/trend) structure exists in the frequency domain?*

> **Reference:** Goerg (2013), *Forecastable Component Analysis (ForeCA).*

In [ ]:
f4 = build_spectral_predictability(air)

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.barh(
    ["Spectral predictability Ω", "Normalised entropy H"],
    [f4.score, f4.normalised_entropy],
    color=[COLORS["ami"], COLORS["pami"]],
)
ax.set_xlim(0, 1)
ax.set_title("F4: Spectral Predictability")
for bar, val in zip(bars, [f4.score, f4.normalised_entropy]):
    ax.text(
        bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}", va="center",
    )
plt.tight_layout()
plt.show()

display(Markdown(f"**Interpretation:** {f4.interpretation}"))

### F5: Largest Lyapunov Exponent (Experimental)

*Does the series exhibit chaotic dynamics?*

> [!CAUTION]
> The LLE diagnostic is experimental.  It requires long, stationary, low-noise series
> and must **not** be used as the sole triage decision-maker.

> **Reference:** Liang et al. (2013), *Permutation Auto-Mutual Information of Electroencephalogram in Anesthesia.*

In [ ]:
lle = triage_result.largest_lyapunov_exponent

if lle is not None:
    lle_rows = {
        "\u03bb estimate": f"{lle.lambda_estimate:.4f}",
        "Embedding dim": lle.embedding_dim,
        "Delay \u03c4": lle.delay,
        "Interpretation": lle.interpretation,
        "Reliability warning": lle.reliability_warning,
    }
    display(pd.DataFrame([lle_rows]).T.rename(columns={0: "F5 \u2014 LLE"}))
else:
    display(Markdown("*F5 was not computed for this series (too short or skipped).*"))

### F6: Entropy-Complexity Band

*Where does the series sit on the complexity spectrum?*

> **Reference:** Ponce-Flores et al. (2020), *Time Series Complexities and Their Relationship to Forecasting Performance.*

In [ ]:
cb = triage_result.complexity_band
assert cb is not None

cb_rows = {
    "Permutation entropy (PE)": f"{cb.permutation_entropy:.3f}",
    "Spectral entropy (SE)": f"{cb.spectral_entropy:.3f}",
    "Complexity band": cb.complexity_band,
    "Interpretation": cb.interpretation,
}
display(pd.DataFrame([cb_rows]).T.rename(columns={0: "F6 \u2014 Complexity Band"}))

# Visual: PE vs SE scatter
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(
    [cb.permutation_entropy], [cb.spectral_entropy],
    s=120, color=COLORS["ami"], zorder=5,
)
ax.annotate(
    "Air Passengers",
    (cb.permutation_entropy, cb.spectral_entropy),
    textcoords="offset points", xytext=(10, 5),
)
ax.set_xlabel("Normalised Permutation Entropy")
ax.set_ylabel("Normalised Spectral Entropy")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("F6: Entropy-Complexity Plane")
plt.tight_layout()
plt.show()

### Diagnostic Dashboard — F1 through F6 summary

In [ ]:
dashboard_rows = [
    {
        "Diagnostic": "F1: Forecastability Profile",
        "Key metric": "Peak horizon",
        "Value": fp.peak_horizon,
        "Verdict": fp.model_now,
    },
    {
        "Diagnostic": "F2: Information Ceiling",
        "Key metric": "Ceiling @ h=1",
        "Value": f"{ceiling[0]:.3f} nats",
        "Verdict": tld.ceiling_summary[:80],
    },
    {
        "Diagnostic": "F3: Learning Curve",
        "Key metric": "Recommended lookback",
        "Value": f"{f3.recommended_lookback} obs",
        "Verdict": "Plateau detected" if f3.plateau_detected else "No plateau",
    },
    {
        "Diagnostic": "F4: Spectral Predictability",
        "Key metric": "Ω score",
        "Value": f"{f4.score:.2f}",
        "Verdict": f4.interpretation[:60],
    },
    {
        "Diagnostic": "F5: Lyapunov Exponent ⚠",
        "Key metric": "λ estimate",
        "Value": f"{lle.lambda_estimate:.4f}" if lle else "N/A",
        "Verdict": (lle.interpretation[:60] if lle else "Not computed"),
    },
    {
        "Diagnostic": "F6: Complexity Band",
        "Key metric": "Band",
        "Value": cb.complexity_band,
        "Verdict": cb.interpretation[:60],
    },
]
display(pd.DataFrame(dashboard_rows))

> **What this tells us:** The six lenses converge — Air Passengers is a *low-complexity,
> spectrally concentrated, highly forecastable* series with strong seasonal structure.
> Models should exploit the 12-month periodicity and the information ceiling is generous.

---
## 4. Same series, different lenses — scorer registry

The `ForecastabilityAnalyzer` supports multiple dependence scorers.  
Let's compare MI, Pearson, Spearman, and distance correlation on the same series.

In [ ]:
registry = default_registry()
scorers = ["mi", "pearson", "spearman", "distance"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
colors_list = [COLORS["ami"], COLORS["pami"], COLORS["sig"], COLORS["med"]]

for i, method in enumerate(scorers):
    analyzer = ForecastabilityAnalyzer(
        n_surrogates=99, random_state=42, method=method,
    )
    result_sc = analyzer.analyze(air, max_lag=24)
    raw_lags_sc = np.arange(1, len(result_sc.raw) + 1)
    partial_lags_sc = np.arange(1, len(result_sc.partial) + 1)
    axes[0].plot(
        raw_lags_sc, result_sc.raw,
        label=f"{method} (raw)", color=colors_list[i], linewidth=1.5,
    )
    axes[1].plot(
        partial_lags_sc, result_sc.partial,
        label=f"{method} (partial)", color=colors_list[i], linewidth=1.5,
    )

axes[0].set_title("Raw dependence by scorer")
axes[0].set_xlabel("Horizon h")
axes[0].legend(fontsize=8)
axes[1].set_title("Partial dependence by scorer")
axes[1].set_xlabel("Horizon h")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

> **What this tells us:** All scorers agree on the seasonal peaks.  MI captures non-linear
> structure that Pearson misses, while distance correlation provides robustness to outliers.

---
## 5. Canonical AMI / pAMI reporting with significance bands

The canonical pipeline adds surrogate-based significance estimation.

In [ ]:
canonical_result = run_canonical_example(
    "air_passengers",
    air,
    max_lag_ami=24,
    max_lag_pami=24,
    n_neighbors=8,
    n_surrogates=99,
    alpha=0.05,
    random_state=42,
)

In [ ]:
lags_c = np.arange(1, len(canonical_result.ami.values) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AMI with bands
ax = axes[0]
ax.plot(lags_c, canonical_result.ami.values, color=COLORS["ami"], linewidth=2, label="AMI")
if canonical_result.ami.upper_band is not None:
    ax.fill_between(
        lags_c,
        canonical_result.ami.lower_band,
        canonical_result.ami.upper_band,
        alpha=0.2, color=COLORS["band"], label="Surrogate band",
    )
if canonical_result.ami.significant_lags is not None:
    sig_lags = np.asarray(canonical_result.ami.significant_lags)
    sig_vals = canonical_result.ami.values[sig_lags - 1]  # lags are 1-based
    ax.scatter(
        sig_lags, sig_vals,
        color=COLORS["sig"], s=40, zorder=5, label="Significant",
    )
ax.set_title("AMI with significance bands")
ax.set_xlabel("Horizon h")
ax.set_ylabel("AMI (nats)")
ax.legend(fontsize=8)

# pAMI with bands
ax = axes[1]
pami_lags_c = np.arange(1, len(canonical_result.pami.values) + 1)
ax.plot(pami_lags_c, canonical_result.pami.values, color=COLORS["pami"], linewidth=2, label="pAMI")
if canonical_result.pami.upper_band is not None:
    ax.fill_between(
        pami_lags_c,
        canonical_result.pami.lower_band,
        canonical_result.pami.upper_band,
        alpha=0.2, color=COLORS["band"], label="Surrogate band",
    )
if canonical_result.pami.significant_lags is not None:
    sig_lags_p = np.asarray(canonical_result.pami.significant_lags)
    sig_vals_p = canonical_result.pami.values[sig_lags_p - 1]  # lags are 1-based
    ax.scatter(
        sig_lags_p, sig_vals_p,
        color=COLORS["sig"], s=40, zorder=5, label="Significant",
    )
ax.set_title("pAMI with significance bands")
ax.set_xlabel("Horizon h")
ax.set_ylabel("pAMI (nats)")
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
# Mediated component plot
mediated = canonical_result.ami.values - canonical_result.pami.values

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lags_c, mediated, color=COLORS["med"], alpha=0.6, label="Mediated = AMI \u2212 pAMI")
ax.set_xlabel("Horizon h")
ax.set_ylabel("Mediated dependence (nats)")
ax.set_title("Mediated Component by Horizon")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Canonical markdown report preview
report_md = build_canonical_markdown(canonical_result)
display(Markdown(report_md[:2000] + "\n\n*...truncated for display...*"))

> **What this tells us:** The significance bands confirm that Air Passengers' AMI and pAMI
> are well above noise at seasonal lags (h=12, 24).  The mediated component tells us
> intermediate horizons carry redundant information.

---
## 6. Robustness — how stable are these results?

Two tests: **backend comparison** (do different pAMI backends agree?) and
**sample-size stress** (does the signal survive with fewer observations?).

In [ ]:
backend_result = run_backend_comparison(
    series_name="air_passengers",
    ts=air,
    max_lag_ami=24,
    max_lag_pami=24,
    backends=["linear_residual", "rf_residual", "extra_trees_residual"],
    n_neighbors=8,
    n_surrogates=99,
    alpha=0.05,
    random_state=42,
)

backend_df = pd.DataFrame(
    [
        {
            "backend": e.backend,
            "directness_ratio": f"{e.directness_ratio:.3f}",
            "auc_ami": f"{e.auc_ami:.3f}",
            "auc_pami": f"{e.auc_pami:.3f}",
            "n_sig_pami": e.n_sig_pami,
        }
        for e in backend_result.entries
    ]
)
display(backend_df)

print(f"Lag ranking stable: {backend_result.lag_ranking_stable}")
print(f"Directness ratio stable: {backend_result.directness_ratio_stable}")

In [ ]:
# Overlay pAMI curves by backend
fig, ax = plt.subplots(figsize=(10, 4))
backend_colors = [COLORS["ami"], COLORS["pami"], COLORS["sig"]]
for i, entry in enumerate(backend_result.entries):
    pami_vals = np.array(entry.pami_values)
    ax.plot(
        np.arange(1, len(pami_vals) + 1), pami_vals,
        label=entry.backend.replace("_", " "),
        color=backend_colors[i % len(backend_colors)],
        linewidth=1.5,
    )
ax.set_xlabel("Horizon h")
ax.set_ylabel("pAMI (nats)")
ax.set_title("pAMI curves across backends")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
stress_result = run_sample_size_stress(
    series_name="air_passengers",
    ts=air,
    fractions=[0.5, 0.75, 1.0],
    max_lag_ami=24,
    max_lag_pami=24,
    n_neighbors=8,
    n_surrogates=99,
    alpha=0.05,
    random_state=42,
)

stress_df = pd.DataFrame(
    [
        {
            "fraction": e.fraction,
            "n_observations": e.n_observations,
            "directness_ratio": f"{e.directness_ratio:.3f}",
            "auc_ami": f"{e.auc_ami:.3f}",
            "auc_pami": f"{e.auc_pami:.3f}",
        }
        for e in stress_result.entries
    ]
)
display(stress_df)
print(f"Directness ratio stable: {stress_result.directness_ratio_stable}")

> **What this tells us:** The pAMI signal is robust across backends and survives even with
> 50% of the data.  If the signal were fragile, we would see divergent curves here.

---
## 7. From diagnostic shape to forecast policy — rolling-origin

The rolling-origin evaluation connects dependence diagnostics to *actual forecast accuracy*.

> **Reference:** Wang et al. (2025), *Time Series Forecastability Measures.*

In [ ]:
rolling_result = run_rolling_origin_evaluation(
    air,
    series_id="air_passengers",
    frequency="Monthly",
    horizons=[1, 3, 6, 12],
    n_origins=5,
    seasonal_period=12,
    random_state=42,
)

In [ ]:
# AMI/pAMI by horizon
horizons_ro = sorted(rolling_result.ami_by_horizon.keys())
ami_vals_ro = [rolling_result.ami_by_horizon[h] for h in horizons_ro]
pami_vals_ro = [rolling_result.pami_by_horizon[h] for h in horizons_ro]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dependence by horizon
x_pos = np.arange(len(horizons_ro))
width = 0.35
axes[0].bar(x_pos - width / 2, ami_vals_ro, width, label="AMI", color=COLORS["ami"])
axes[0].bar(x_pos + width / 2, pami_vals_ro, width, label="pAMI", color=COLORS["pami"])
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f"h={h}" for h in horizons_ro])
axes[0].set_title("Dependence by horizon (rolling-origin)")
axes[0].set_ylabel("MI (nats)")
axes[0].legend()

# sMAPE by model
for fr in rolling_result.forecast_results:
    smapes = [fr.smape_by_horizon[h] for h in horizons_ro]
    axes[1].plot(horizons_ro, smapes, "o-", label=fr.model_name, linewidth=1.5)
axes[1].set_xlabel("Horizon h")
axes[1].set_ylabel("sMAPE")
axes[1].set_title("Forecast accuracy by model")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
# Winner table
winner_rows = []
for h in horizons_ro:
    best_model = min(
        rolling_result.forecast_results, key=lambda fr: fr.smape_by_horizon[h],
    )
    winner_rows.append(
        {
            "Horizon": h,
            "Best model": best_model.model_name,
            "sMAPE": f"{best_model.smape_by_horizon[h]:.2f}",
            "AMI": f"{rolling_result.ami_by_horizon[h]:.3f}",
            "pAMI": f"{rolling_result.pami_by_horizon[h]:.3f}",
        }
    )
display(pd.DataFrame(winner_rows))

> **What this tells us:** Rolling-origin connects information-theoretic diagnostics to
> forecast accuracy.  Horizons with higher AMI tend to have lower sMAPE.

---
## 8. Air Passengers in portfolio context — batch triage

Compare Air Passengers against a portfolio of structurally different series.

In [ ]:

def showcase_batch_router(request, readiness):
    return MethodPlan(
        route="univariate_no_significance",
        compute_surrogates=False,
        assumptions=["Showcase batch preview skips surrogate bands for speed."],
        rationale="Notebook-friendly fast panel ranking.",
    )


batch_req = BatchTriageRequest(
    items=[
        BatchSeriesRequest(series_id="air_passengers", series=air.tolist()),
        BatchSeriesRequest(
            series_id="white_noise",
            series=generate_white_noise(n_samples=500, random_state=42).tolist(),
        ),
        BatchSeriesRequest(
            series_id="ar1_strong",
            series=generate_ar1(n_samples=500, phi=0.9, random_state=42).tolist(),
        ),
        BatchSeriesRequest(
            series_id="henon_map",
            series=generate_henon_map(n_samples=500).tolist(),
        ),
        BatchSeriesRequest(
            series_id="stock_returns",
            series=generate_simulated_stock_returns(n_samples=500, random_state=42).tolist(),
        ),
    ],
    max_lag=24,
    random_state=42,
)
batch_response = run_batch_triage(
    batch_req,
    triage_runner=lambda request: run_triage(request, router=showcase_batch_router),
)

In [ ]:
# Summary table
batch_rows = []
for item in batch_response.items:
    batch_rows.append(
        {
            "rank": item.rank,
            "series": item.series_id,
            "forecastability": item.forecastability_class or "N/A",
            "directness": item.directness_class or "N/A",
            "complexity_band": item.complexity_band_label or "N/A",
            "recommended_next_action": item.recommended_next_action,
        }
    )
display(pd.DataFrame(batch_rows).sort_values("rank"))

In [ ]:
# Ranked scatter: forecastability class vs directness ratio
fig, ax = plt.subplots(figsize=(8, 5))

palette = {"high": COLORS["ami"], "medium": COLORS["pami"], "low": COLORS["sig"]}
for item in batch_response.items:
    fc = item.forecastability_class or "low"
    dr = item.directness_ratio or 0.0
    ax.scatter(
        dr, item.rank,
        s=120, color=palette.get(fc, COLORS["band"]), zorder=5,
    )
    ax.text(dr + 0.015, item.rank, item.series_id, va="center", fontsize=9)

ax.set_xlabel("Directness ratio")
ax.set_ylabel("Rank (1 = highest priority)")
ax.set_title("Batch Triage — 5-Series Portfolio")
ax.invert_yaxis()
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

> **What this tells us:** Air Passengers stands out in the top-right (high forecastability,
> high directness), while white noise / stock returns occupy the bottom-left.
> Batch triage gives a portfolio-level view at a glance.

---
## 9. Interpretation patterns A–E

The package classifies every series into one of five **interpretation patterns**.

| Pattern | AMI | pAMI | Meaning |
|:-------:|:---:|:----:|:--------|
| **A** | high | high | Rich structured dependence — complex models justified |
| **B** | high | low/medium | Mediated dependence — compact models preferred |
| **C** | medium | — | Uncertain — seasonal or regularised models |
| **D** | low | low | Both weak — baseline methods only |
| **E** | high | — (sMAPE ≥ naive) | Exploitability mismatch — investigate |

In [ ]:
pattern_series = {
    "air_passengers": air,
    "white_noise": generate_white_noise(n_samples=500, random_state=42),
    "ar1_strong": generate_ar1(n_samples=500, phi=0.9, random_state=42),
    "henon_map": generate_henon_map(n_samples=500),
    "stock_returns": generate_simulated_stock_returns(n_samples=500, random_state=42),
}

pattern_rows = []
for name, series in pattern_series.items():
    result = run_triage(TriageRequest(series=series, max_lag=24, random_state=42))
    if result.interpretation is not None:
        pattern_rows.append(
            {
                "series": name,
                "forecastability_class": result.interpretation.forecastability_class,
                "directness_class": result.interpretation.directness_class,
                "modeling_regime": result.interpretation.modeling_regime.replace("_", " "),
                "primary_lags": ", ".join(
                    f"h={h}" for h in result.interpretation.primary_lags[:5]
                ),
            }
        )

display(pd.DataFrame(pattern_rows))

> [!TIP]
> The `modeling_regime` field maps directly to patterns A–E.  Use it to drive automated
> model-selection pipelines.

---
## 10. Exogenous driver screening

F8 answers: *which candidate drivers add genuine incremental predictive value
beyond what the target's own lags provide?*

We use a **deterministic stub evaluator** for fast, reproducible demonstration.

In [ ]:
_HORIZONS_EXOG = list(range(1, 11))


def _build_driver_profiles() -> dict[str, dict[int, tuple[float, float, float]]]:
    """Deterministic per-horizon exogenous dependence profiles."""
    return {
        "strong_driver": {
            1: (0.28, 0.22, 0.79),
            2: (0.26, 0.20, 0.77),
            3: (0.24, 0.19, 0.79),
            4: (0.22, 0.17, 0.77),
            5: (0.19, 0.14, 0.74),
            6: (0.17, 0.12, 0.71),
            7: (0.14, 0.10, 0.71),
            8: (0.12, 0.08, 0.67),
            9: (0.10, 0.06, 0.60),
            10: (0.08, 0.05, 0.62),
        },
        "weak_driver": {
            1: (0.04, 0.02, 0.50),
            2: (0.03, 0.02, 0.67),
            3: (0.03, 0.01, 1.40),
            4: (0.02, 0.01, 1.50),
            5: (0.02, 0.01, 1.50),
            6: (0.02, 0.01, 1.70),
            7: (0.01, 0.01, 1.20),
            8: (0.01, 0.01, 1.60),
            9: (0.01, 0.01, 1.40),
            10: (0.01, 0.01, 1.80),
        },
        "redundant_driver": {
            1: (0.18, 0.12, 0.67),
            2: (0.17, 0.11, 0.65),
            3: (0.16, 0.10, 0.63),
            4: (0.15, 0.10, 0.67),
            5: (0.14, 0.09, 0.64),
            6: (0.12, 0.08, 0.67),
            7: (0.11, 0.07, 0.64),
            8: (0.10, 0.06, 0.60),
            9: (0.08, 0.05, 0.62),
            10: (0.07, 0.04, 0.57),
        },
    }


def _build_stub_pair_evaluator(
    *, profiles: dict[str, dict[int, tuple[float, float, float]]],
):
    """Deterministic evaluator for fast demo without multiprocessing."""

    def _pair_evaluator(
        target,
        exog,
        *,
        case_id,
        target_name,
        exog_name,
        horizons,
        n_origins,
        random_state,
        n_surrogates,
        min_pairs_raw,
        min_pairs_partial,
        analysis_scope,
        project_extension,
    ) -> ExogenousBenchmarkResult:
        profile = profiles[exog_name]
        warning_horizons = [h for h in horizons if profile[h][2] > 1.0]
        return ExogenousBenchmarkResult(
            case_id=case_id,
            target_name=target_name,
            exog_name=exog_name,
            horizons=horizons,
            raw_cross_mi_by_horizon={h: profile[h][0] for h in horizons},
            conditioned_cross_mi_by_horizon={h: profile[h][1] for h in horizons},
            directness_ratio_by_horizon={h: profile[h][2] for h in horizons},
            origins_used_by_horizon={h: 6 for h in horizons},
            warning_horizons=warning_horizons,
            metadata={"stubbed": 1},
        )

    return _pair_evaluator

In [ ]:
profiles = _build_driver_profiles()
exog_config = ExogenousScreeningWorkbenchConfig.model_validate(
    {
        "horizons": _HORIZONS_EXOG,
        "n_origins": 6,
        "random_state": 42,
        "n_surrogates": 99,
        "min_pairs_raw": 10,
        "min_pairs_partial": 10,
        "redundancy_alpha": 0.55,
        "apply_bh_correction": True,
        "bh_fdr_alpha": 0.10,
        "lag_windows": [
            {"name": "near_term", "start_horizon": 1, "end_horizon": 3},
            {"name": "mid_term", "start_horizon": 4, "end_horizon": 7},
            {"name": "long_term", "start_horizon": 8, "end_horizon": 10},
        ],
        "pruning": {
            "enabled": True,
            "min_mean_usefulness": 0.012,
            "min_peak_usefulness": 0.020,
            "horizon_usefulness_floor": 0.012,
            "min_horizons_above_floor": 2,
        },
        "recommendation": {
            "keep_min_mean_usefulness": 0.095,
            "keep_min_peak_usefulness": 0.130,
            "review_min_mean_usefulness": 0.040,
            "review_min_peak_usefulness": 0.060,
        },
    }
)

n_samples_exog = 220
target_exog = np.linspace(0.0, 1.0, n_samples_exog, dtype=float)
drivers = {
    name: np.linspace(idx, idx + 1.0, n_samples_exog, dtype=float)
    for idx, name in enumerate(sorted(profiles), start=1)
}

exog_result = run_exogenous_screening_workbench(
    target_exog,
    drivers,
    target_name="air_passengers_demo",
    config=exog_config,
    pair_evaluator=_build_stub_pair_evaluator(profiles=profiles),
)

In [ ]:
# Driver summary table
driver_rows = driver_summary_table_rows(exog_result)
display(
    pd.DataFrame(driver_rows)[
        [
            "overall_rank", "driver_name", "recommendation",
            "mean_usefulness_score", "peak_usefulness_score",
            "top_horizon", "bh_significant", "redundancy_score",
        ]
    ]
)

In [ ]:
# Horizon usefulness table (first 15 rows)
horizon_rows_exog = horizon_usefulness_table_rows(exog_result)
display(
    pd.DataFrame(horizon_rows_exog)[
        ["driver_name", "horizon", "usefulness_score", "directness_ratio"]
    ].head(15)
)

> **What this tells us:** The screening workbench separates strong drivers (keep) from
> noisy ones (reject) using horizon-by-horizon usefulness scoring.  Redundant drivers
> are flagged via correlation analysis.

---
## 11. Agent-ready serialisation

Every triage result can be serialised to a JSON bundle for downstream consumption
by LLM agents, dashboards, or data pipelines.

In [ ]:
bundle = build_triage_result_bundle(triage_result)
payload = bundle.model_dump(mode="json")

# Show top-level structure
print("Bundle top-level keys and types:")
print(json.dumps({k: type(v).__name__ for k, v in payload.items()}, indent=2))

In [ ]:
# Interpretation fields
interp = triage_result.interpretation
assert interp is not None

display(
    Markdown(
        f"### Interpretation fields\n\n"
        f"- **Forecastability class:** {interp.forecastability_class}\n"
        f"- **Directness class:** {interp.directness_class}\n"
        f"- **Modeling regime:** {interp.modeling_regime}\n"
        f"- **Primary lags:** {interp.primary_lags}\n"
        f"- **Narrative:** {interp.narrative}\n"
    )
)

> [!TIP]
> Use `save_triage_result_bundle()` to persist the bundle to disk.  The JSON is
> schema-versioned and backward-compatible.

---
## 12. Generated report

The canonical markdown report summarises all key findings in a publication-ready format.

In [ ]:
report = build_canonical_markdown(canonical_result)
display(Markdown(report))

In [ ]:
# Final summary dashboard compiling all sections
final_summary = [
    {
        "Section": "Triage",
        "Finding": (
            f"Forecastability = {view.forecastability_class}, "
            f"pattern = {view.pattern_class}"
        ),
    },
    {
        "Section": "F1 Profile",
        "Finding": (
            f"Peak at h={fp.peak_horizon}, "
            f"{len(fp.informative_horizons)} informative horizons"
        ),
    },
    {"Section": "F2 Ceiling", "Finding": tld.ceiling_summary[:80]},
    {
        "Section": "F3 Learning Curve",
        "Finding": (
            f"Lookback = {f3.recommended_lookback}, "
            f"plateau = {f3.plateau_detected}"
        ),
    },
    {"Section": "F4 Spectral", "Finding": f"\u03a9 = {f4.score:.2f}"},
    {
        "Section": "F5 Lyapunov",
        "Finding": (
            f"\u03bb = {lle.lambda_estimate:.4f}" if lle else "Not computed"
        ),
    },
    {"Section": "F6 Complexity", "Finding": f"Band = {cb.complexity_band}"},
    {
        "Section": "Robustness",
        "Finding": (
            f"Backend stable = {backend_result.lag_ranking_stable}, "
            f"stress stable = {stress_result.directness_ratio_stable}"
        ),
    },
    {
        "Section": "Rolling-origin",
        "Finding": (
            f"{len(rolling_result.forecast_results)} models evaluated "
            f"across {len(horizons_ro)} horizons"
        ),
    },
    {
        "Section": "Batch triage",
        "Finding": f"{len(batch_response.items)} series compared",
    },
    {
        "Section": "Exog screening",
        "Finding": f"{len(exog_result.driver_summaries)} drivers ranked",
    },
]

display(Markdown("### All-Methods Summary Dashboard"))
display(pd.DataFrame(final_summary))

---
## 13. Product surface map — where to go next

The `forecastability` package exposes the same diagnostics through multiple surfaces:

| Surface | Entry point | When to use |
|:--------|:------------|:------------|
| **Python API** | `run_triage()`, `run_batch_triage()` | Notebooks, scripts, pipelines |
| **CLI** | `uv run python -m forecastability.adapters.cli` | Terminal automation, CI/CD |
| **Dashboard** | `forecastability.adapters.dashboard` | Interactive exploration |
| **MCP server** | `forecastability.adapters.mcp_server` | LLM tool-use integration |
| **PydanticAI agent** | `forecastability.adapters.pydantic_ai_agent` | Agentic workflows |

### Other notebooks

- `notebooks/walkthroughs/` — step-by-step walkthroughs of individual features
- `notebooks/triage/` — triage-specific deep dives
- `examples/univariate/` — runnable single-series scripts for CI validation
- `examples/covariant_informative/` — runnable multivariate, exogenous, directional, and causal-discovery scripts for CI validation

> [!TIP]
> Start with `run_triage()` for a single series, then scale to `run_batch_triage()`
> for a portfolio.  Use the rolling-origin evaluation to connect diagnostics to forecast accuracy.